# 02 — Tiền xử lý (chạy 1 lần cho cả nhóm)

Thu thập 3 dataset → loại trùng (MD5 + dHash 256 bit) → nhóm ảnh gần trùng → chia `split_mixed` /
`split_external` + 5 fold CV trên train → mask Otsu (kênh S) → lưu ảnh 224×224 + `manifest.csv` + `stats.json`.

**Kaggle:** Add Input 3 dataset `anhnguyen7508/bananaid`, `anhnguyen7508/bananaimagebd`,
`kimlunl/bananaripeness-github`; bật Internet. Chạy hết → **Save Version** → ở tab Output chọn
**New Dataset** (vd. tên `banana-preprocessed`) để cả nhóm Add Input dùng chung.

Code: `src/banana_ripeness/preprocessing/`. Notebook chỉ gọi hàm.

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Kaggle: bật Settings -> Internet = On. Code lấy từ GitHub, dữ liệu lấy từ Kaggle Dataset (Add Input).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "luan"          # tên người chạy, ghi vào bảng kết quả chung

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "banana_ripeness").exists()), None)
if local is not None:                       # chạy trên máy, trong thư mục repo
    CODE_DIR = local
else:                                       # Kaggle / Colab: clone vào /tmp (không lẫn vào output notebook)
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
os.environ["BANANA_OWNER"] = OWNER
commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code:", CODE_DIR, "| commit", commit or "?")

%load_ext autoreload
%autoreload 2

In [ ]:
from banana_ripeness import load_config
from banana_ripeness.preprocessing import run_preprocessing

cfg = load_config(CODE_DIR / "configs" / "kaggle.yaml")   # đường dẫn 3 dataset nằm trong file này
# Nếu Kaggle gắn dataset ở đường dẫn khác, sửa: cfg.dataset_dirs["BananaID"] = "/kaggle/input/..."
for k, v in cfg.dataset_dirs.items():
    print(f"{k:24s} {v}  {'OK' if Path(v).exists() else '!! KHÔNG THẤY'}")

In [ ]:
result = run_preprocessing(cfg)

## Bảng kiểm tra

In [ ]:
for name in ["eda_check", "summary_before_after", "dedup_log", "hash64_vs_256", "split_mixed",
             "split_external", "split_external_dataset", "leakage", "hue_masked", "mask_stats"]:
    if name in result.tables:
        print(f"\n===== {name} =====")
        display(result.tables[name])

## Hình QA (mask Otsu, cặp ảnh nghi trùng khác nhãn, so sánh hash 64 vs 256 bit)

In [ ]:
from IPython.display import Image, display
for name, path in result.figures.items():
    print(name, path)
    display(Image(filename=str(path)))

## Cảnh báo cần báo nhóm

In [ ]:
for w in result.warnings:
    print("-", w)
print("Output:", result.output_dir)